# GAN 예비 생성 이미지 - FID·LPIPS 점검

## 목표
이미 생성해 둔 네 조건의 사진을 대상으로 **참고용** FID와 LPIPS를 계산합니다. 이 노트북은 GAN이나 분류기를 다시 학습하지 않습니다. 실제 사진과 생성 사진을 각각 조건별 50장씩 사용합니다.

**주의:** 실제 사진은 해당 GAN의 학습 사진이므로 독립 품질 검증이 아닙니다. 일부가 과거 평가 구간에 속해 있어 이번 결과로 후속 분류기 평가나 최종 모델 선택을 하면 안 됩니다. 새 실험은 평가 구간을 먼저 분리하고 GAN을 다시 학습하거나 새로운 독립 출력으로 평가해야 합니다. 원본 데이터는 각 클래스별 출력 1회입니다.

FID는 실제와 생성 사진 묶음의 특징 분포 차이입니다(낮을수록 유사). 표준 Inception 2048차원 특징을 사용하므로 실제/생성 각각 50장인 이번 점수는 특히 불안정한 예비값입니다. 서로 다른 기준 사진을 쓰는 깨끗한 사진과 마스킹 사진 간의 점수만으로 전처리 우열을 단정하지 않습니다.

LPIPS는 두 이미지의 특징 차이입니다. 정답 쌍이 없는 이번 GAN에 실제와 생성 사진을 임의로 1:1 맞춰 정확도라고 하지 않고, **서로 다른 생성 사진 100쌍의 거리**와 **생성 사진별 학습 사진 중 가장 가까운 거리**를 따로 계산합니다. 마스킹 영역 및 일반 이미지 사전학습 특징에 영향받을 수 있고, 낮은 거리가 복제를 증명하거나 큰 거리가 좋은 품질을 보장하지는 않습니다.

실행 시 Colab에 `torchmetrics`, `torch-fidelity`, `lpips`를 설치하고 공식 사전학습 가중치를 다운로드합니다. 기존 실험 폴더는 읽기만 하고 결과는 새 Drive 폴더에 기록합니다. G4를 선택한 뒤 전체 실행하세요. 예외가 발생해도 Drive 동기화와 런타임 자동 반납을 시도하지만, 강제 종료 시 수동 확인이 필요합니다.


In [ ]:
from google.colab import drive, runtime
from pathlib import Path
import sys, subprocess, traceback, json, os, time, hashlib, random, csv, zipfile, itertools

mounted = False
run_root = None
try:
    # 이전 결과는 읽기만 하며 별도 폴더에 새 결과를 만듭니다.
    drive.mount('/content/drive')
    mounted = True
    root = Path('/content/drive/MyDrive/산공종설')
    source = root / 'results/gan_four_conditions_20261101_run01'
    run_root = root / 'results/gan_quality_metrics_four_conditions_20261101_run01'
    if not source.is_dir():
        raise FileNotFoundError(f'기존 GAN 결과 폴더가 없습니다: {source}')
    if run_root.exists() and any(run_root.iterdir()):
        raise FileExistsError(f'기존 결과를 덮어쓰지 않도록 중단합니다: {run_root}')
    run_root.mkdir(parents=True, exist_ok=True)
    log_dir = run_root / 'logs'; log_dir.mkdir()
    install_log = log_dir / 'install.log'
    print('평가 의존 패키지를 설치합니다. 기존 학습 결과는 변경하지 않습니다.', flush=True)
    cmd = [sys.executable, '-m', 'pip', 'install', 'torchmetrics==1.8.2', 'torch-fidelity==0.3.0', 'lpips==0.1.4']
    with install_log.open('w', encoding='utf-8') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
        for line in p.stdout:
            f.write(line); f.flush()
        if p.wait() != 0:
            raise RuntimeError(f'패키지 설치 실패: {install_log}')
    import numpy as np
    import torch
    import torchvision
    from PIL import Image, ImageOps, ImageDraw
    import lpips
    import torchmetrics
    from torchmetrics.image.fid import FrechetInceptionDistance

    if not torch.cuda.is_available():
        raise RuntimeError('GPU가 없습니다. G4를 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    if 'RTX PRO 6000' not in gpu.upper():
        raise RuntimeError(f'선택한 GPU가 G4가 아닙니다: {gpu}')
    device = torch.device('cuda')
    print('GPU:', gpu, flush=True)
    seed = 42
    groups = ['clean_normal', 'clean_underfill', 'partial_normal', 'partial_underfill']
    settings = {'seed': seed, 'gpu': gpu, 'source': str(source), 'n_real': 50, 'n_generated': 50,
                'image_format': 'RGB PNG 256x256; real and generated equally loaded',
                'fid': 'torchmetrics FrechetInceptionDistance feature=2048 normalize=True (torch-fidelity Inception)',
                'lpips': 'lpips 0.1.4 alex pretrained, RGB float [-1,1]',
                'lpips_generated_pairs': '100 distinct unordered pairs sampled without replacement seed=42 separately per group',
                'lpips_nearest': 'each generated image against all 50 GAN training images, minimum',
                'limitations': 'Training real reference; 50 images; some prior held-out frames included; single print per class; preliminary only.',
                'versions': {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'torchmetrics':torchmetrics.__version__, 'numpy':np.__version__}}
    (run_root/'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
    summary_rows, pair_rows, nearest_rows = [], [], []
    review_dir = run_root/'review'; review_dir.mkdir()

    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
        return h.hexdigest()

    def load_images(paths):
        samples = []
        for path in paths:
            with Image.open(path) as im:
                if im.size != (256, 256) or im.mode != 'RGB':
                    raise ValueError(f'예상과 다른 입력: {path} / {im.mode} / {im.size}')
                a = np.asarray(im, dtype=np.uint8).copy()
            samples.append(torch.from_numpy(a).permute(2,0,1))
        return torch.stack(samples)

    # LPIPS의 AlexNet과 보정 가중치를 다운로드할 수 있습니다.
    lp_model = lpips.LPIPS(net='alex', verbose=False).eval().to(device)
    for param in lp_model.parameters(): param.requires_grad_(False)
    input_hashes = []
    for group in groups:
        print(f'[{group}] 입력 확인 및 점검 중', flush=True)
        real_dir = source/'prepared'/'data'/group
        fake_dir = source/'gan'/group/'generated'
        real_files, fake_files = sorted(real_dir.glob('*.png')), sorted(fake_dir.glob('*.png'))
        if len(real_files) != 50 or len(fake_files) != 50:
            raise RuntimeError(f'{group}: 실사 {len(real_files)}장, 생성 {len(fake_files)}장. 각 50장 필요')
        for kind, paths in (('real_train',real_files),('generated',fake_files)):
            input_hashes.extend({'group':group,'type':kind,'filename':p.name,'sha256':sha(p)} for p in paths)
        real = load_images(real_files)
        fake = load_images(fake_files)
        # 표준 FID: 두 묶음에 같은 입력 규칙을 적용합니다.
        fid_score, fid_status = None, 'ok'
        try:
            fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
            for start in range(0,50,16):
                fid.update(real[start:start+16].to(device).float()/255, real=True)
                fid.update(fake[start:start+16].to(device).float()/255, real=False)
            with torch.no_grad(): fid_value = float(fid.compute().detach().cpu())
            if not np.isfinite(fid_value): raise ValueError(f'FID가 유한하지 않습니다: {fid_value}')
            fid_score = fid_value
            del fid
        except Exception as exc:
            fid_status = f'failed: {type(exc).__name__}: {exc}'
            print('FID 계산 실패:', fid_status, flush=True)
        # LPIPS용 범위 [-1,1]. 생성 100쌍과 50x50 최근접 이미지 검색.
        rr = real.to(device).float()/127.5-1.0
        ff = fake.to(device).float()/127.5-1.0
        pairs = random.Random(seed).sample(list(itertools.combinations(range(50),2)),100)
        pair_values = []
        with torch.no_grad():
            for start in range(0,100,16):
                p = pairs[start:start+16]
                a = torch.tensor([x for x,y in p], device=device)
                b = torch.tensor([y for x,y in p], device=device)
                vals = lp_model(ff[a],ff[b]).flatten().detach().cpu().tolist()
                for (i,j), value in zip(p,vals):
                    if not np.isfinite(value): raise ValueError('LPIPS 생성 쌍 값이 유한하지 않음')
                    pair_values.append(float(value))
                    pair_rows.append({'group':group,'generated_a':fake_files[i].name,'generated_b':fake_files[j].name,'lpips':value})
            group_nearest=[]
            for i in range(50):
                vals=[]
                for start in range(0,50,16):
                    chunk=rr[start:start+16]
                    distances=lp_model(ff[i:i+1].expand(len(chunk),-1,-1,-1),chunk).flatten()
                    vals.extend(distances.detach().cpu().tolist())
                if not np.all(np.isfinite(vals)): raise ValueError('LPIPS 최근접 값이 유한하지 않음')
                j=int(np.argmin(vals))
                item={'group':group,'generated':fake_files[i].name,'nearest_training_real':real_files[j].name,'lpips':float(vals[j])}
                group_nearest.append(item); nearest_rows.append(item)
        # 가까운 순으로 최대 8쌍. 후보일 뿐 복제 판정이 아닙니다.
        examples=sorted(group_nearest,key=lambda x:x['lpips'])[:8]
        tile, gap, header = 160, 10, 32
        board=Image.new('RGB',(4*(2*tile+gap)+3*gap,2*(tile+header)+gap),'white')
        draw=ImageDraw.Draw(board)
        for k,item in enumerate(examples):
            x=(k%4)*(2*tile+2*gap); y=(k//4)*(tile+header+gap)
            with Image.open(fake_dir/item['generated']) as im:
                board.paste(ImageOps.fit(im.convert('RGB'),(tile,tile)),(x,y))
            with Image.open(real_dir/item['nearest_training_real']) as im:
                board.paste(ImageOps.fit(im.convert('RGB'),(tile,tile)),(x+tile+gap,y))
            draw.text((x,y+tile+3),f"{k+1}: generated | train, LPIPS={item['lpips']:.4f}",fill='black')
        board.save(review/f'{group}_nearest_8.jpg',quality=94)
        nearest_values=[x['lpips'] for x in group_nearest]
        summary_rows.append({'group':group,'n_train_real':50,'n_generated':50,'FID_2048':fid_score,'FID_status':fid_status,
                             'LPIPS_100_generated_pairs_mean':float(np.mean(pair_values)),
                             'LPIPS_100_generated_pairs_median':float(np.median(pair_values)),
                             'LPIPS_nearest_train_mean':float(np.mean(nearest_values)),
                             'LPIPS_nearest_train_median':float(np.median(nearest_values)),
                             'LPIPS_nearest_train_min':float(np.min(nearest_values))})
        print(f'[{group}] 완료. FID 상태: {fid_status}; LPIPS 이미지 검토 필요', flush=True)
        del rr,ff,real,fake
        torch.cuda.empty_cache()
    def writecsv(name, rows):
        with (run_root/name).open('w',encoding='utf-8-sig',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    writecsv('summary.csv',summary_rows);writecsv('lpips_generated_pairs.csv',pair_rows)
    writecsv('lpips_nearest_training.csv',nearest_rows);writecsv('input_sha256.csv',input_hashes)
    print('조건별 예비 수치:',json.dumps(summary_rows,ensure_ascii=False,indent=2),flush=True)
    (run_root/'README_결과_읽기.md').write_text("""# 읽는 방법

summary.csv의 FID는 실제 GAN 학습 사진 50장과 생성 50장의 표준 FID 예비값이며 작을수록 특징 분포가 가깝습니다. 표본이 작고 서로 다른 조건의 실제 기준 사진이 달라 조건 간 숫자로 성능 순위를 주장할 수 없습니다.

생성 이미지 간 LPIPS 100쌍은 다양성 참고값이고, 생성별 학습 사진 최근접 LPIPS는 복제 의심 여부를 육안 확인할 후보를 제공합니다. 노이즈나 마스킹이 점수에 영향을 줄 수 있으며 어떤 거리도 복제 여부나 좋은 품질의 증거는 아닙니다. review 이미지는 가까운 순 8쌍이며 왼쪽 생성, 오른쪽 실제 학습 사진입니다.

이번 GAN은 기존 검증·평가 구간 사진도 일부 포함하므로 생성 품질의 독립 검증 또는 분류기 성능 검증으로 사용할 수 없습니다. 후속 실험은 학습 구간에서 GAN을 다시 학습하거나 독립된 새 출력으로 평가해야 합니다. 정상과 언더필은 각각 출력 1회입니다. 입력 이미지·파일 지문과 설정·버전·랜덤 시드는 별도 파일에 보존됩니다.
""",encoding='utf-8')
    zip_path=run_root/'gan_quality_metrics_results.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in run_root.rglob('*'):
            if p.is_file() and p != zip_path: z.write(p,p.relative_to(run_root).as_posix())
    with zipfile.ZipFile(zip_path) as z:
        bad=z.testzip()
        if bad: raise RuntimeError(f'ZIP 손상: {bad}')
    print('점검 완료. 참고 결과:',zip_path,flush=True)
except BaseException as exc:
    trace=traceback.format_exc()
    print('점검 실패:',trace,flush=True)
    if run_root is not None:
        try:
            run_root.mkdir(parents=True,exist_ok=True)
            (run_root/'error.log').write_text(trace,encoding='utf-8')
        except Exception: pass
finally:
    if mounted:
        try: drive.flush_and_unmount()
        except BaseException as exc: print('Drive 동기화 확인 필요:',exc,flush=True)
    print('런타임 반납 요청',flush=True)
    try: runtime.unassign()
    except BaseException as exc: print('수동 반납 확인 필요:',exc,flush=True)


## 결과 확인

Drive의 `산공종설/results/gan_quality_metrics_four_conditions_20261101_run01/`에서 CSV·검토 이미지·ZIP을 확인하세요. 실제 실행 전에는 결과가 없습니다.